# Data Loading - Build & Push to HuggingFace Hub

This notebook loads the Sci-ImageMiner competition data, builds cleaned
datasets, and pushes them to HuggingFace Hub for downstream training.


In [ ]:
import subprocess
from pathlib import Path
from staged_qwen3_5_scivqa.data import (
    build_vqa_dataset,
    build_summary_dataset,
    build_table_dataset,
    load_vqa_dataset,
    load_summary_dataset,
    load_table_dataset,
    load_vqa_from_hub,
    load_summary_from_hub,
    load_table_from_hub,
)
from staged_qwen3_5_scivqa.config import HF_VQA_REPO, HF_SUMMARY_REPO, HF_TABLE_REPO
from huggingface_hub import login as hf_login

In [ ]:
# Clone competition data (skip if already cloned)
repo_url = "https://github.com/sciknoworg/ALD-E-ImageMiner/"
dest = Path.cwd().parent / "ALD-E-ImageMiner"

if not dest.exists():
    subprocess.run(["git", "clone", repo_url, str(dest)], check=True)
else:
    print(f"Already cloned: {dest}")

In [ ]:
hf_login()

## 1. Build & Push to HuggingFace Hub

We build cleaned DatasetDict objects with typed Features and push them
to HuggingFace Hub. Three separate repos: VQA, Summary, Table.

In [ ]:
# Build VQA dataset (all answer types)
dd_vqa = build_vqa_dataset(
    categories=["train", "dev"],
    push_to_hub=True,
    repo_id=HF_VQA_REPO,
)
print(f"VQA splits: {list(dd_vqa.keys())}")

In [ ]:
# Build Summary dataset
dd_summary = build_summary_dataset(
    categories=["train", "dev"],
    push_to_hub=True,
    repo_id=HF_SUMMARY_REPO,
)
print(f"Summary splits: {list(dd_summary.keys())}")

In [ ]:
# Build Table extraction dataset
dd_table = build_table_dataset(
    categories=["train", "dev"],
    push_to_hub=True,
    repo_id=HF_TABLE_REPO,
)
print(f"Table splits: {list(dd_table.keys())}")

## 2. Load from HuggingFace Hub

Once pushed, any machine with internet can load the cleaned datasets.

In [ ]:
# Load VQA dataset from HF Hub (returns a Dataset)
vqa_ds = load_vqa_from_hub(repo_id=HF_VQA_REPO, split="train")
print(f"VQA train samples: {len(vqa_ds)}")
print(f"Columns: {vqa_ds.column_names}")
print(f"Answer types: {set(vqa_ds['answer_type'])}")

In [ ]:
# Load Summary dataset from HF Hub
summary_ds = load_summary_from_hub(repo_id=HF_SUMMARY_REPO, split="train")
print(f"Summary train samples: {len(summary_ds)}")
print(f"Columns: {summary_ds.column_names}")

In [ ]:
# Load Table dataset from HF Hub
table_ds = load_table_from_hub(repo_id=HF_TABLE_REPO, split="train")
print(f"Table train samples: {len(table_ds)}")
print(f"Columns: {table_ds.column_names}")

## 3. Backward-Compatible Loading

The existing load_vqa_dataset, load_summary_dataset, and load_table_dataset
functions now accept source="hf" to load from HF Hub instead of local files.
They return Unsloth conversation dicts ready for training.

In [ ]:
# Load as Unsloth conversation samples from HF (backward-compatible path)
samples, valid, invalid = load_vqa_dataset(
    "train",
    answer_types=["Factoid"],
    source="hf",
    hf_repo_id=HF_VQA_REPO,
)
print(f"Factoid samples: {len(samples)} (valid={valid}, invalid={invalid})")
print(f"Keys in first sample: {list(samples[0].keys())}")

In [ ]:
# Load Summary as Unsloth conversation samples from HF
samples_s, valid_s, invalid_s = load_summary_dataset(
    "train", source="hf", hf_repo_id=HF_SUMMARY_REPO
)
print(f"Summary samples: {len(samples_s)}")

## 4. Verify Token Statistics

Confirm the loaded samples have reasonable token lengths for your model.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from transformers import AutoProcessor
from staged_qwen3_5_scivqa.analysis import calculate_token_stats
from staged_qwen3_5_scivqa.config import MODEL_ID

processor = AutoProcessor.from_pretrained(MODEL_ID)
df_tokens = calculate_token_stats(samples, processor)
display(
    df_tokens[["assistant_tokens", "total_tokens"]].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_tokens["assistant_tokens"].hist(bins=30, ax=axes[0], color="salmon", edgecolor="black")
axes[0].set_title("Assistant Tokens")
df_tokens["total_tokens"].hist(bins=30, ax=axes[1], color="skyblue", edgecolor="black")
axes[1].set_title("Total Sequence Tokens")
plt.show()